In [11]:
!pip install -q --upgrade langchain langchain-core langchain-community langchain-groq ddgs

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [15]:
import warnings
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from ddgs import DDGS

# ✅ LLM (Groq - very fast)
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast + free tier friendly
    temperature=0.2
)

# ✅ Tool 1: Calculator
@tool
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

# ✅ Tool 2: Web Search
@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        return "\n".join([r["body"] for r in results])
    except Exception as e:
        return f"Search error: {str(e)}"

tools = [calculator, web_search]

# ✅ Strong ReAct Prompt
system_prompt = """
You are a strict ReAct AI agent.

Follow this format strictly:

Thought: think step by step
Action: tool name (if needed)
Action Input: input
Observation: result
... repeat if needed ...

Final Answer: always give a clear final answer.

Rules:
- Use tools when needed
- Always end with Final Answer
- Final Answer must be plain text only
- Do NOT use LaTeX, markdown, or special formatting
- Do NOT wrap answers in \\boxed{} or symbols
"""

# ✅ Create Agent (LATEST API)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [17]:
import re

def clean_output(text):
    if not text:
        return "[No answer]"

    # Extract only content after "Final Answer:"
    match = re.search(r"Final Answer:\s*(.*)", text, re.IGNORECASE)
    if match:
        text = match.group(1)

    # Remove LaTeX \boxed{}
    text = re.sub(r'\\boxed\{(.+?)\}', r'\1', text)

    # Remove $ symbols
    text = text.replace("$", "")

    # Remove extra "Final Answer:" repetitions
    text = re.sub(r"Final Answer:\s*", "", text, flags=re.IGNORECASE)

    return text.strip()

while True:
    query = input("\nAsk something (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": query}
        ]
    })

    # ✅ Extract final text safely
    final_msg = response["messages"][-1].content

    if isinstance(final_msg, list):
        text_output = ""
        for block in final_msg:
            if isinstance(block, dict) and block.get("type") == "text":
                text_output += block.get("text", "")
    else:
        text_output = str(final_msg)

    print("\nFinal Answer:\n", clean_output(text_output) or "[No text returned]")


Ask something (type 'exit' to quit): 5 + 6

Final Answer:
 11

Ask something (type 'exit' to quit): What is weather in Kochi today?

Final Answer:
 The current weather in Kochi today is expected to be Mist with a maximum temperature of 30.9°C and a minimum of 26.2°C.

Ask something (type 'exit' to quit): What is Agentic AI?

Final Answer:
 Agentic AI is a type of artificial intelligence that is capable of taking initiative, making decisions, and acting independently to achieve its goals.

Ask something (type 'exit' to quit): exit
